Installing libraries and dependencies

In [5]:
%pip install qiskit qiskit-aer matplotlib numpy

   ---------------------------------------- 0.0/9.8 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.8 MB 14.9 MB/s eta 0:00:01
   -------- ------------------------------- 2.1/9.8 MB 14.9 MB/s eta 0:00:01
   -------- ------------------------------- 2.1/9.8 MB 14.9 MB/s eta 0:00:01
   ------------ --------------------------- 3.1/9.8 MB 3.7 MB/s eta 0:00:02
   ----------------- ---------------------- 4.2/9.8 MB 4.1 MB/s eta 0:00:02
   ------------------------- -------------- 6.3/9.8 MB 5.1 MB/s eta 0:00:01
   ---------------------------------- ----- 8.4/9.8 MB 5.5 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.8 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 9.8/9.8 MB 5.3 MB/s  0:00:01
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -------- ------------------------------- 2.1/9.6 MB 17.2 MB/s eta 0:00:01
   -------- ------------------------------- 2.1/9.6 MB 17.2 MB/s eta 0:00:01
   -------- ------------


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import qiskit
import numpy as np
import math
from math import sqrt
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit import transpile
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
from qiskit.circuit.library import RZGate
from qiskit.visualization import array_to_latex
from qiskit.circuit.library import RYGate

Matplotlib is building the font cache; this may take a moment.


## Dividing a 2^n vector into two 2^n-1 subvectors and treating complex angles' magnitudes

While the subvector is larger than 1, it will be recursively split into two other subvectors of equal size. After the last recursive call, the probability values are computed into a new vector angles, which is returned as a vector of magnitudes


In [7]:
def gen_angles(x):
    if len(x) > 1:
        new_x = list(range(len(x)//2))

        for k in range(len(new_x)):
            new_x[k] = sqrt(x[2*k]**2 + x[2*k+1]**2)

        inner_angles = gen_angles(new_x)

        angles = list(range(len(x)//2))
        for k in range(len(angles)):
            if new_x[k] != 0:
                if x[2*k] > 0:
                    angles[k] = 2 * np.arcsin(x[2*k+1] / new_x[k])
                else:
                    angles[k] = 2 * np.pi - 2 * np.arcsin(x[2*k+1] / new_x[k])
            else:
             angles[k] = 0

        angles = inner_angles + angles
        return angles 
    else:
        return []               

## Validating the generation of the normalized magnitudes vector

In [8]:
target_angles = [sqrt(0.03), sqrt(0.07), sqrt(0.15), sqrt(0.05), sqrt(0.1), sqrt(0.3),
                 sqrt(0.2), sqrt(0.1)]

target_angles_normalized = gen_angles(target_angles)
print(target_angles_normalized)

[np.float64(1.9823131728623846), np.float64(1.9106332362490186), np.float64(1.4274487578895316), np.float64(1.9823131728623846), np.float64(1.0471975511965979), np.float64(2.0943951023931957), np.float64(1.2309594173407747)]


## Preparing and normalizing complex angles' phases

In [9]:
def gen_angles_z(x):
    if len(x) > 1:
        new_x = list(range(len(x)//2))

        for k in range(0, len(new_x)):
            new_x[k] = (x[2*k] + x[2*k+1])/2

        inner_angles = gen_angles_z(new_x)

        angles_z = list(range(len(x)//2))

        for k in range(0, len(new_x)):
            angles_z[k] = x[2*k+1] - x[2*k]

        angles_z = inner_angles + angles_z
        return angles_z

    else:
        return []
    

## Validating the gen_angles_z function with a 2-qubit vector

In [10]:
phases = [ np.sqrt(0.1) * np.exp(1j * 0),
    np.sqrt(0.3) * np.exp(1j * np.pi / 4),
    np.sqrt(0.2) * np.exp(1j * np.pi / 2),
    np.sqrt(0.4) * np.exp(1j * 3 * np.pi / 4)]

target_phases = np.angle(phases)

target_phases_normalized = gen_angles_z(target_phases)

print(target_phases_normalized)

[np.float64(1.5707963267948966), np.float64(0.7853981633974483), np.float64(0.7853981633974483)]


# Implementation of gen_circuit

The `gen_circuit` function builds the state preparation quantum circuit based on the angles computed by the `gen_angles` function.

In [11]:
def gen_circuit(angles, angles_z = None):

    N = len(angles) + 1
    n = int(math.log2(N))
    circuit = QuantumCircuit(n)
    q = circuit.qubits[::-1]

    def level(k):
        return int(math.log2(k + 1))

    def index(k, j, q, circuit):
        s = k - (2**j - 1)
        for i in range(j):
            bit = (s >> (j - 1 - i)) & 1
            if bit == 0:
                circuit.x(q[i])

    for k in range(N - 1):  
        j = level(k)
        
        index(k, j, q, circuit)
        angle = angles[k]
        if j == 0:
            circuit.ry(angle, q[0])

            if angles_z is not None:
                circuit.rz(angles_z[k], q[0])
        else:
            controlled_ry = RYGate(angle).control(num_ctrl_qubits=j)
            circuit.append(controlled_ry, [*q[:j], q[j]])

            if angles_z is not None:
                # É necessário garantir a importação no topo do arquivo: from qiskit.circuit.library import RZGate
                controlled_rz = RZGate(angles_z[k]).control(num_ctrl_qubits=j)
                circuit.append(controlled_rz, [*q[:j], q[j]])

            
        index(k, j, q, circuit)

    return circuit

# Testing and Validation of the gen_circuit Function

Let's instantiate the `gen_circuit` function with the angles from the 8-dimensional example (3 qubits) described in Section 2 of the paper to visually check the generated structure.

In [12]:

test_angles = [1.98, 1.91, 1.43, 1.98, 1.05, 2.09, 1.23]

qc = gen_circuit(test_angles)


print("Quantum Circuit Structure Output (N=8, 3 qubits):")
print(qc.draw(output='text'))

Quantum Circuit Structure Output (N=8, 3 qubits):
                                                        ┌──────────┐          »
q_0: ───────────────────────────────────────────────────┤ Ry(1.98) ├──────────»
                      ┌──────────┐     ┌──────────┐┌───┐└────┬─────┘┌───┐     »
q_1: ─────────────────┤ Ry(1.91) ├─────┤ Ry(1.43) ├┤ X ├─────■──────┤ X ├─────»
     ┌──────────┐┌───┐└────┬─────┘┌───┐└────┬─────┘├───┤     │      ├───┤┌───┐»
q_2: ┤ Ry(1.98) ├┤ X ├─────■──────┤ X ├─────■──────┤ X ├─────■──────┤ X ├┤ X ├»
     └──────────┘└───┘            └───┘            └───┘            └───┘└───┘»
«     ┌──────────┐     ┌──────────┐     ┌──────────┐
«q_0: ┤ Ry(1.05) ├─────┤ Ry(2.09) ├─────┤ Ry(1.23) ├
«     └────┬─────┘┌───┐└────┬─────┘┌───┐└────┬─────┘
«q_1: ─────■──────┤ X ├─────■──────┤ X ├─────■──────
«          │      ├───┤     │      └───┘     │      
«q_2: ─────■──────┤ X ├─────■────────────────■──────
«                 └───┘                             


# Circuit loading 

Writing the `main` function responsible for the `load_circuit` step

In [13]:
def load_circuit(angles, angles_z=None):
    M = len(angles)  # N - 1 qubits
    qc = QuantumCircuit(M)

    # Lines 2-3: Ry on every qubit
    for k in range(M):
        qc.ry(angles[k], k)

    # Lines 4-5: Rz on every qubit (phase information)
    if angles_z is not None:
        for k in range(M):
            qc.rz(angles_z[k], k)

    # Lines 6-14: combine subtrees bottom-up with controlled swaps
    left = lambda i: 2 * i + 1
    right = lambda i: 2 * i + 2

    actual = (M - 2) // 2  # parent(N - 2)
    while actual >= 0:
        left_index = left(actual)
        right_index = right(actual)
        while right_index < M:
            qc.cswap(actual, left_index, right_index)
            left_index = left(left_index)
            right_index = left(right_index)
        actual -= 1

    return qc

# Self test

In [14]:
def data_qubits(M):
    """Indices of the data register (left spine of the tree): 0, 1, 3, 7, ..."""
    n = int(math.log2(M + 1))
    return [2 ** j - 1 for j in range(n)]

 
def _data_probabilities(qc):
    from qiskit.quantum_info import Statevector
 
    spine = data_qubits(qc.num_qubits)
    n = len(spine)
    probs = Statevector(qc).probabilities(spine)  # first qarg = least significant bit
    out = np.zeros(2 ** n)
    for idx, p in enumerate(probs):
        k = int(format(idx, f"0{n}b")[::-1], 2)  # qubit 0 -> most significant bit
        out[k] = p
    return out
 
 
def _path_phase_sum(omega, k):
    """Sum of +/- lambda/2 along the root->leaf path of leaf k (minus = left child)."""
    n = int(math.log2(len(omega)))
    total, level_vals = 0.0, list(omega)
    # build per-level lambdas bottom-up, then walk top-down
    levels = []
    while len(level_vals) > 1:
        levels.append([level_vals[2 * i + 1] - level_vals[2 * i] for i in range(len(level_vals) // 2)])
        level_vals = [(level_vals[2 * i] + level_vals[2 * i + 1]) / 2 for i in range(len(level_vals) // 2)]
    for depth in range(n):  # depth 0 = root
        lev = levels[n - 1 - depth]
        node = k >> (n - depth)  # ancestor index at this level
        bit = (k >> (n - 1 - depth)) & 1
        total += (lev[node] / 2) * (1 if bit else -1)
    return total
 
 
def _selftest():
    rng = np.random.default_rng(0)
 
    # Paper example (N = 8)
    p = [0.03, 0.07, 0.15, 0.05, 0.1, 0.3, 0.2, 0.1]
    x = np.sqrt(p)
    got = _data_probabilities(load_circuit(gen_angles(list(x))))
    assert np.allclose(got, p), (got, p)
    print("paper example N=8 : OK")
 
    # Random real, non-negative and signed vectors
    for N in (4, 8, 16):
        for signed in (False, True):
            v = rng.random(N) + 0.05
            if signed:
                v *= rng.choice([-1, 1], N)
            v /= np.linalg.norm(v)
            got = _data_probabilities(load_circuit(gen_angles(list(v))))
            assert np.allclose(got, v ** 2), (N, signed)
    print("random real N=4,8,16 (with and without signs): OK")
 
    # Complex: magnitudes through the circuit
    for N in (4, 8, 16):
        mag = rng.random(N) + 0.05
        mag /= np.linalg.norm(mag)
        omega = rng.uniform(-np.pi, np.pi, N)
        qc = load_circuit(gen_angles(list(mag)), gen_angles_z(list(omega)))
        got = _data_probabilities(qc)
        assert np.allclose(got, mag ** 2), N
        # phases: omega_k - mean(omega) must equal the sum of +/-lambda/2 on the path
        for k in range(N):
            assert np.isclose(omega[k] - omega.mean(), _path_phase_sum(omega, k)), (N, k)
    print("complex N=4,8,16 (magnitudes on circuit, phase decomposition classical): OK")
 
    # Depth growth
    for N in (8, 16, 32, 64):
        qc = load_circuit(gen_angles(list(np.ones(N) / sqrt(N))))
        print(f"N={N:3d}  qubits={qc.num_qubits:3d}  depth={qc.depth()}")
 
 
if __name__ == "__main__":
    _selftest()

paper example N=8 : OK
random real N=4,8,16 (with and without signs): OK
complex N=4,8,16 (magnitudes on circuit, phase decomposition classical): OK
N=  8  qubits=  7  depth=4
N= 16  qubits= 15  depth=7
N= 32  qubits= 31  depth=11
N= 64  qubits= 63  depth=16
